In [8]:
import pandas as pd
import numpy as np

data = {
    'patient_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
                   111, 112, 113, 114, 115, 101, 107, 118, 119, 120],
    'age': ['25', '34', None, '45', '29', None, '38', '52', '27', '41',
            '33', 'unknown', '48', '26', '35', '25', '38', '31', None, '44'],
    'weight': ['70', '65', '80', None, '75', None, '68', '90', '72', '85',
               '78', None, '82', '69', 'N/A', '70', '68', '74', None, '88'],
    'blood_pressure': [120, 130, None, 140, 125, None, 135, None, 118, 145,
                      128, None, 138, 122, None, 120, 135, 126, None, 142],
    'medication': ['Aspirin', 'Metformin', 'Lisinopril', None, 'Aspirin',
                   'Metformin', 'Lisinopril', 'Aspirin', None, 'Metformin',
                   'Lisinopril', 'Aspirin', None, 'Metformin', 'Aspirin',
                   'Aspirin', 'Lisinopril', 'Metformin', 'Aspirin', None],
    'insurance_provider': ['Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', None,
                          'Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', 'Blue Cross',
                          'Aetna', None, 'UnitedHealth', 'Blue Cross', 'Aetna',
                          'Blue Cross', 'Aetna', 'Cigna', 'UnitedHealth', None]
}

df = pd.DataFrame(data)

In [9]:
print("--- Info ---")
df.info()

print("\n--- Missing Values Count ---")
print(df.isnull().sum())

print("\n--- Missing Values Percentage ---")
print((df.isnull().sum() / len(df)) * 100)

print("\n--- Duplicate Rows Count ---")
print(df.duplicated().sum())

--- Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   patient_id          20 non-null     int64  
 1   age                 17 non-null     object 
 2   weight              16 non-null     object 
 3   blood_pressure      14 non-null     float64
 4   medication          16 non-null     object 
 5   insurance_provider  17 non-null     object 
dtypes: float64(1), int64(1), object(4)
memory usage: 1.1+ KB

--- Missing Values Count ---
patient_id            0
age                   3
weight                4
blood_pressure        6
medication            4
insurance_provider    3
dtype: int64

--- Missing Values Percentage ---
patient_id             0.0
age                   15.0
weight                20.0
blood_pressure        30.0
medication            20.0
insurance_provider    15.0
dtype: float64

--- Duplicate Rows Count ---
2


In [10]:
# Convert to numeric (errors='coerce' turns strings like 'unknown' or 'N/A' into NaN)
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['weight'] = pd.to_numeric(df['weight'], errors='coerce')

# Convert categorical data
df['insurance_provider'] = df['insurance_provider'].astype('category')

print("\n--- New Missing Values After Conversion ---")
print(df[['age', 'weight']].isnull().sum())

print("\n--- Updated Data Types ---")
print(df.dtypes)



--- New Missing Values After Conversion ---
age       4
weight    5
dtype: int64

--- Updated Data Types ---
patient_id               int64
age                    float64
weight                 float64
blood_pressure         float64
medication              object
insurance_provider    category
dtype: object


In [11]:
# Numerical: Use Median
df['age'] = df['age'].fillna(df['age'].median())
df['weight'] = df['weight'].fillna(df['weight'].median())
df['blood_pressure'] = df['blood_pressure'].fillna(df['blood_pressure'].median())

# Categorical: Medication (Still a string/object, so fillna works normally)
df['medication'] = df['medication'].fillna(df['medication'].mode()[0])

# Categorical: Insurance (Handling the Category Type constraint)
df['insurance_provider'] = df['insurance_provider'].cat.add_categories("Unknown")
df['insurance_provider'] = df['insurance_provider'].fillna('Unknown')

print("\n--- Missing Values after Cleaning ---")
print(df.isnull().sum())


--- Missing Values after Cleaning ---
patient_id            0
age                   0
weight                0
blood_pressure        0
medication            0
insurance_provider    0
dtype: int64


In [12]:
print("Shape before removing duplicates:", df.shape)

# View all duplicates based on patient_id
print("\n--- Duplicate Records Based on Patient ID ---")
print(df[df.duplicated(subset=['patient_id'], keep=False)])

# Remove duplicates
df = df.drop_duplicates(subset=['patient_id'], keep='first')

print("\nShape after removing duplicates:", df.shape)

Shape before removing duplicates: (20, 6)

--- Duplicate Records Based on Patient ID ---
    patient_id   age  weight  blood_pressure  medication insurance_provider
0          101  25.0    70.0           120.0     Aspirin         Blue Cross
6          107  38.0    68.0           135.0  Lisinopril              Aetna
15         101  25.0    70.0           120.0     Aspirin         Blue Cross
16         107  38.0    68.0           135.0  Lisinopril              Aetna

Shape after removing duplicates: (18, 6)


In [13]:
# 1. Start Fresh
df_clean = pd.DataFrame(data).copy()
initial_shape = df_clean.shape

# 2. Types
df_clean['age'] = pd.to_numeric(df_clean['age'], errors='coerce')
df_clean['weight'] = pd.to_numeric(df_clean['weight'], errors='coerce')
df_clean['insurance_provider'] = df_clean['insurance_provider'].astype('category')

# 3. Missing Values
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['weight'] = df_clean['weight'].fillna(df_clean['weight'].median())
df_clean['blood_pressure'] = df_clean['blood_pressure'].fillna(df_clean['blood_pressure'].median())
df_clean['medication'] = df_clean['medication'].fillna(df_clean['medication'].mode()[0])
df_clean['insurance_provider'] = df_clean['insurance_provider'].cat.add_categories("Unknown").fillna("Unknown")

# 4. Duplicates
df_clean = df_clean.drop_duplicates(subset=['patient_id'], keep='first')

# --- FINAL REPORT ---
print("=== VERIFICATION REPORT ===")
print(f"Shape: {initial_shape} -> {df_clean.shape}")
print(f"Total Missing: {df_clean.isnull().sum().sum()}")
print(f"Total Duplicates: {df_clean.duplicated().sum()}")
print("\nFinal Data Types:")
print(df_clean.dtypes)


=== VERIFICATION REPORT ===
Shape: (20, 6) -> (18, 6)
Total Missing: 0
Total Duplicates: 0

Final Data Types:
patient_id               int64
age                    float64
weight                 float64
blood_pressure         float64
medication              object
insurance_provider    category
dtype: object
